In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

sys.path.append(str(SRC_DIR))



In [2]:
from load_data import(
    load_hpo
)

from build_triples import(
    load_filtered_hpo_annotations,
    build_disease_hpo_triples,
    build_entity_metadata,
    save_triples,
    save_entity_metadata,
    get_triple_statistics,
)

from train_transe_pykeen import(
    create_triples_factory,
    split_triples_factory,
    train_transe_model,
    save_embedding,
    save_metrics,
)

/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
HPO_PATH = RAW_DIR / "hp.obo"
HPOA_FILTERED_PATH = PROCESSED_DIR / "hpoa_filtered.csv"

hpo = load_hpo(HPO_PATH)
hpoa_filtered = load_filtered_hpo_annotations(HPOA_FILTERED_PATH)

print("HPO nodes:", hpo.number_of_nodes())
print("HPO edges:", hpo.number_of_edges())
print("Filtered annotation rows:", len(hpoa_filtered))
print("Filtered diseases:", hpoa_filtered["database_id"].nunique())
print("Filtered HPO terms:", hpoa_filtered["hpo_id"].nunique())

hpoa_filtered.head()

HPO nodes: 19389
HPO edges: 23677
Filtered annotation rows: 20178
Filtered diseases: 1000
Filtered HPO terms: 4569


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [7]:
triples = build_disease_hpo_triples(
    hpoa_filtered=hpoa_filtered,
    hpo=hpo,
    include_hpo_hierarchy=True,
    include_only_relevant_hpo_edges=True,
)

triple_stats = get_triple_statistics(triples)

triple_stats

{'triples_total': 31521,
 'entities_total': 11760,
 'relations_total': 2,
 'relation_has_phenotype': 20132,
 'relation_is_a': 11389}

In [9]:
triples.head(20)

,head,relation,tail
0,OMIM:619340,has_phenotype,HP:0011097
1,OMIM:619340,has_phenotype,HP:0002187
2,OMIM:619340,has_phenotype,HP:0001518
3,OMIM:619340,has_phenotype,HP:0032792
4,OMIM:619340,has_phenotype,HP:0011451
5,OMIM:619340,has_phenotype,HP:0010851
6,OMIM:619340,has_phenotype,HP:0001789
7,OMIM:619340,has_phenotype,HP:0200134
8,OMIM:619340,has_phenotype,HP:0002643
9,OMIM:619426,has_phenotype,HP:0000286


In [10]:
triples["relation"].value_counts()

relation
has_phenotype    20132
is_a             11389
Name: count, dtype: int64

In [13]:
entity_metadata = build_entity_metadata(
    hpoa_filtered=hpoa_filtered,
    hpo=hpo,
    triples=triples,
)

print("Entities:", len(entity_metadata))
print(entity_metadata["entity_type"].value_counts())

entity_metadata.head(20)

Entities: 11760
entity_type
phenotype    10760
disease       1000
Name: count, dtype: int64


,entity_id,entity_type,label
0,DECIPHER:21,disease,Miller-Dieker syndrome (MDS)
1,DECIPHER:3,disease,Williams-Beuren Syndrome (WBS)
2,DECIPHER:45,disease,Xq28 (MECP2) duplication
3,DECIPHER:54,disease,Angelman syndrome (Type 2)
4,DECIPHER:81,disease,15q26 overgrowth syndrome
5,HP:0000002,phenotype,Abnormality of body height
6,HP:0000003,phenotype,Multicystic kidney dysplasia
7,HP:0000008,phenotype,Abnormal morphology of female internal genitalia
8,HP:0000009,phenotype,Functional abnormality of the bladder
9,HP:0000010,phenotype,Recurrent urinary tract infections


In [14]:
TRIPLES_PATH = PROCESSED_DIR / "triples_disease_hpo.tsv"
ENTITY_METADATA_PATH = PROCESSED_DIR / "entity_metadata_disease_hpo.csv"

save_triples(triples, TRIPLES_PATH)
save_entity_metadata(entity_metadata, ENTITY_METADATA_PATH)

print("Saved triples to:", TRIPLES_PATH)
print("Saved metadata to:", ENTITY_METADATA_PATH)

Saved triples to: /workspaces/GraphRepresentationLearning/data/processed/triples_disease_hpo.tsv
Saved metadata to: /workspaces/GraphRepresentationLearning/data/processed/entity_metadata_disease_hpo.csv


In [18]:
triples_factory = create_triples_factory(
    triples=triples,
    create_inverse_triples=True,
)

print("Entities:", triples_factory.num_entities)
print("Relations:", triples_factory.num_relations)
print("Triples:", triples_factory.num_triples)

Entities: 11760
Relations: 4
Triples: 31521


In [22]:
training, validation, testing = split_triples_factory(
    triples_factory=triples_factory,
    ratios=(0.8, 0.1, 0.1),
    random_state=5,
)

print("Training triples:", training.num_triples)
print("Validation triples:", validation.num_triples)
print("Testing triples:", testing.num_triples)

Training triples: 25216
Validation triples: 3152
Testing triples: 3153
